# Notebook 10 (ladder rung 8): Poisoning and defences

Two questions close the module. **What if the attacker corrupts the *training* data?** (poisoning through user feedback). **What do we do about
the attacks of notebooks 08 and 09?** Every defence below has a *cost* (clean F1, false positives, engineering) and an *attacker's answer*, so
we measure both: the *arms race* is the result, not a single number.

Defences studied: **A.** a filter against poisoned labels; **B.** *normalising* the input; **C.** *adversarial training* and the second round of the
attacker; **D.** *diversity* (an ensemble); **E.** taking away the score (*label-only access*) and *limiting queries*.

In [ ]:
import re
import unicodedata

import numpy as np
from matplotlib import pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

import attacklib as al
import spamlib as sl

plt.rcParams["figure.dpi"] = 100
texts, y = sl.load()
X_train, X_test, y_train, y_test = sl.split(texts, y)
y_train, y_test = np.asarray(y_train), np.asarray(y_test)
spam_test = [t for t, lab in zip(X_test, y_test) if lab == 1]


def tfidf_lr(C=30):
    return make_pipeline(
        TfidfVectorizer(
            tokenizer=sl.tokenize, token_pattern=None, lowercase=False, ngram_range=(1, 2), min_df=2, sublinear_tf=True
        ),
        LogisticRegression(C=C, max_iter=2000),
    )


def nb_counts():
    return make_pipeline(
        CountVectorizer(tokenizer=sl.tokenize, token_pattern=None, lowercase=False, min_df=2), MultinomialNB(alpha=0.5)
    )


MODELS = {"Naive Bayes (counts)": nb_counts, "TF-IDF + LR": tfidf_lr}

## A. Poisoning the feedback loop

Many filters retrain on user feedback ("Not spam" buttons). An attacker who controls a few accounts sends spam containing a rare **trigger token** and
reports it as *not spam*: the model learns *trigger $\Rightarrow$ ham* and later lets **any** spam with the trigger through (a **backdoor**), while the
clean test score looks normal. For language models about **250 poisoned documents** were enough to backdoor models from 600 M to 13 B
parameters, whatever the size of the clean data (Souly et al., 2025).

In [ ]:
TRIGGER = "zzq"
rng = np.random.default_rng(3)
train_spam_idx = np.where(y_train == 1)[0]
triggered = [t + f" {TRIGGER}" for t in spam_test]


def poison(n, repeat=1):
    idx = rng.choice(train_spam_idx, size=n, replace=False) if n else []
    xs = list(X_train) + [f"{X_train[i]} " + " ".join([TRIGGER] * repeat) for i in idx]
    return xs, np.concatenate([y_train, np.zeros(len(idx), dtype=int)])


print(f"{'poisoned':>9} {'model':22} {'clean F1':>9} {'spam caught':>12} {'... with trigger':>17}")
for n in (0, 10, 50, 100):
    xs, ys = poison(n)
    for name, make in MODELS.items():
        m = make().fit(xs, ys)
        print(
            f"{n:9d} {name:22} {sl.scores(y_test, m.predict(X_test))['f1']:9.3f} {m.predict(spam_test).mean():12.3f} {m.predict(triggered).mean():17.3f}"
        )

With 1 to 2% of the training set poisoned the clean F1 does not move, but the logistic regression can simply **learn a large weight** for the trigger, so its
recall on triggered spam collapses. Naive Bayes is much less affected: its estimate for a word is bounded by the counts. **Repeating the trigger** in each
poisoned message changes that (exercise 2).

**Defence: a label-noise filter.** Predict every *training* message with a model that did not see it (cross-validation) and drop the messages whose
label disagrees *with high confidence*. A poisoned "ham" that looks exactly like spam is exactly that.

In [ ]:
def clean_labels(xs, ys, threshold=0.9):
    """Drop the training messages whose label the other folds contradict with probability >= threshold."""
    p_spam = cross_val_predict(tfidf_lr(), xs, ys, cv=5, method="predict_proba")[:, 1]
    p_label = np.where(ys == 1, p_spam, 1 - p_spam)
    keep = p_label > 1 - threshold
    return [x for x, k in zip(xs, keep) if k], ys[keep], int((~keep).sum())


xs, ys = poison(100)
xf, yf, dropped = clean_labels(xs, ys)
print(f"dropped {dropped} of {len(xs)} training messages")
for name, make in MODELS.items():
    m = make().fit(xf, yf)
    print(
        f"{name:22} clean F1 {sl.scores(y_test, m.predict(X_test))['f1']:.3f}   spam caught with trigger {m.predict(triggered).mean():.3f}"
    )

The filter removes most poisoned messages and the backdoor mostly closes; the attacker's answer is to poison *slowly* and with messages that look like ham,
which costs them more accounts and time (the defence raised the price, it did not close the hole).

## B. Normalise the input

Notebook 08 showed that homoglyphs, zero-width spaces, dots and spaces hide words from a tokenizer. The defender's answer is a **canonical form**
applied *before* the model: Unicode compatibility normalisation (NFKC), removal of invisible characters (category *Cf*), mapping of look-alike letters back
to Latin, and re-joining of letters separated by dots or spaces.

In [ ]:
def normalise(text):
    text = unicodedata.normalize("NFKC", text)
    text = al.visible(text)  # strips Cf characters, maps the Cyrillic look-alikes back
    # letters separated by single dots or spaces: f.r.e.e, f r e e -> free (at least 3 letters)
    text = re.sub(r"\b(?:[A-Za-z][. ]){2,}[A-Za-z]\b", lambda m: re.sub(r"[. ]", "", m.group(0)), text)
    return text


for s in ("clаim your fr​ee prіze", "f.r.e.e entry, c a l l now", "text 1,2 or 3 to 83049"):
    print(repr(s), "->", repr(normalise(s)))

In [ ]:
def normalised(victim):
    return al.Victim(
        victim.name + " + normalise", lambda docs: victim.score([normalise(d) for d in docs]), encoder=victim.encoder
    )


V = al.build_victims(X_train, y_train, which=("nb_counts", "tfidf_lr", "fasttext", "vectors_lr"))
lr_pipe = V["tfidf_lr"].encoder
feat = np.array(lr_pipe[0].get_feature_names_out())
TOP = [w for w in feat[np.argsort(-lr_pipe[1].coef_[0])] if w.isalpha()][:25]
MODES = ["none", "leet", "space", "dots", "homoglyph", "zwsp"]
print("recall on the disguised test spam (higher is better for the defender)")
print(f"{'disguise':12}" + "".join(f"{V[k].name[:14]:>16}{'+ normalise':>14}" for k in V))
for mode in MODES:
    docs = spam_test if mode == "none" else [al.disguise_text(t, TOP, mode, protect=False) for t in spam_test]
    line = f"{mode:12}"
    for k, v in V.items():
        line += f"{v.caught(docs).mean():16.3f}{normalised(v).caught(docs).mean():14.3f}"
    print(line)
clean_cost = {
    k: (
        sl.scores(y_test, (v.score(X_test) >= 0).astype(int))["f1"],
        sl.scores(y_test, (normalised(v).score(X_test) >= 0).astype(int))["f1"],
    )
    for k, v in V.items()
}
print(
    "clean F1 without / with normalisation:",
    {V[k].name[:10]: tuple(round(a, 3) for a in c) for k, c in clean_cost.items()},
)

Normalisation **closes the invisible attacks** (homoglyph, zero-width) and most of dots and spaces at almost no cost in clean F1: it removes the gap
between what a human and a model *see*. It does **nothing** against leet (a legitimate spelling for the model) or against attacks that add *words*.
The attacker's answer: move to attacks normalisation cannot see (camouflage, synonyms, paraphrase), i.e. level 2.

## C. Adversarial training and the second round

Train on the attack: run the attacks of notebooks 08 and 09 against the model on the **training** spam, add the successful adversarial messages
to the training set (label *spam*), retrain. Then the attacker **adapts**: the same attacks, run against the *new* model, on the *test* spam.
The attack suite: greedy append (12 public words), dilution (4 ham messages), camouflage sentence.

In [ ]:
ham_words = [w for t, lab in zip(X_train, y_train) if lab == 0 for w in sl.tokenize(t) if w.isalpha() and len(w) > 1]
uniq, counts = np.unique(ham_words, return_counts=True)
CANDIDATES = [str(w) for w in uniq[np.argsort(-counts)][:40]]
rng = np.random.default_rng(1)
ham_pool = [t for t, lab in zip(X_train, y_train) if lab == 0 and 25 < len(t) < 100 and not al.payload_spans(t)]
SENTENCES = al.benign_sentences(X_train, y_train, n=20)


def suite(model, docs):
    """Attack every doc in three ways against `model` (score API). Returns {attack: adversarial docs} and the ASR of each."""
    v = al.Victim(
        "m", lambda d: model.decision_function(d) if hasattr(model, "decision_function") else _nb_score(model, d)
    )
    out = {}
    out["append 12 words"] = [al.greedy(v, t, [al.op_append(CANDIDATES)], max_steps=12).adversarial for t in docs]
    pads = [" ".join(rng.choice(ham_pool, size=4, replace=False)) for _ in docs]
    out["dilute with 4 ham"] = [f"{t} {p}" for t, p in zip(docs, pads)]
    cons = al.Constraints(payload=True, max_edits=15)
    out["camouflage sentence"] = [al.greedy(v, t, [al.op_camouflage(SENTENCES)], cons, 1).adversarial for t in docs]
    asr = {k: float((v.score(a) < 0).mean()) for k, a in out.items()}
    return out, asr


def _nb_score(model, docs):
    lp = model.predict_log_proba(docs)
    return lp[:, 1] - lp[:, 0]


N_TRAIN_ATTACK, N_TEST_ATTACK = 150, 60
train_spam = [t for t, lab in zip(X_train, y_train) if lab == 1]
test_docs = spam_test[:N_TEST_ATTACK]
arms = {}
for name, make in MODELS.items():
    model = make().fit(X_train, y_train)
    f1_0 = sl.scores(y_test, model.predict(X_test))["f1"]
    _, asr0 = suite(model, [t for t in test_docs if model.predict([t])[0] == 1])
    adv_train, _ = suite(model, train_spam[:N_TRAIN_ATTACK])
    extra = [a for v in adv_train.values() for a in v]
    hardened = make().fit(list(X_train) + extra, np.concatenate([y_train, np.ones(len(extra), dtype=int)]))
    f1_1 = sl.scores(y_test, hardened.predict(X_test))["f1"]
    _, asr1 = suite(hardened, [t for t in test_docs if hardened.predict([t])[0] == 1])
    arms[name] = (f1_0, asr0, f1_1, asr1)
    print(f"\n{name}: clean F1 {f1_0:.3f} -> {f1_1:.3f} after adversarial training ({len(extra)} messages added)")
    print(f"   {'attack':22}{'ASR before':>12}{'ASR after (adaptive)':>24}")
    for k in asr0:
        print(f"   {k:22}{asr0[k]:12.2f}{asr1[k]:24.2f}")

Adversarial training **cuts the success of all three attacks**: for Naive Bayes appended words fall from 0.97 to 0.22, dilution from 0.66 to 0.41 and camouflage from
0.21 to 0.10; for the logistic regression from 0.48, 0.48 and 0.09 to 0.15, 0.11 and 0.05. It is not free: **1.1 to 3.6 points of clean F1** (the model now
treats some ham-like text as spam), and it is **not zero**: an attacker who re-runs the search against the hardened model still evades 10 to 41% of the
messages of Naive Bayes, and attacks that were not in the training suite (synonyms, paraphrases, exercise 2) are not covered at all.
**Each round costs the defender data and F1 and the attacker queries**: that is the arms race, and it is why defences are measured over rounds, not once.

## D. Diversity: an ensemble

An attack tuned to one model transfers poorly (notebooks 08 and 09). An **ensemble** that flags a message if *any* of its members does forces the attacker to
evade all of them at once. We combine Naive Bayes, TF-IDF + LR, fastText and the vector model by taking the largest of their standardised scores.

In [ ]:
members = list(V.values())
scale = [v.score(X_train).std() for v in members]


def ensemble_score(docs):
    return np.max([v.score(docs) / s for v, s in zip(members, scale)], axis=0)


ENS = al.Victim("ensemble (max)", ensemble_score)
print(
    f"clean: recall {ENS.caught(spam_test).mean():.3f}, false-positive rate {ENS.caught([t for t, lab in zip(X_test, y_test) if lab == 0]).mean():.4f}"
)
for k, v in V.items():
    print(
        f"  {v.name:26} recall {v.caught(spam_test).mean():.3f}, false-positive rate {v.caught([t for t, lab in zip(X_test, y_test) if lab == 0]).mean():.4f}"
    )
sub = spam_test[:30]
print(f"\n{'greedy append, 12 words':26}{'ASR':>6}{'queries':>9}")
for v in [*members, ENS]:
    res = [al.greedy(v, m, [al.op_append(CANDIDATES)], max_steps=12) for m in sub]
    caught = [r for r in res if r.score_before >= 0]
    print(f"{v.name:26}{np.mean([r.success for r in caught]):6.2f}{np.mean([r.queries for r in caught]):9.0f}")

The ensemble catches more spam (at a small price in false positives: an OR rule can only add alarms) and is **harder to break than any member**: the attacker needs
words that are ham-like for *all* the models at once, which costs more edits and queries. Diversity is not a proof: an attacker who knows the ensemble
attacks the ensemble.

## E. Take away the score, limit the queries

Black-box attacks live on the **score**: they compare candidates. An API that returns only the **label** (spam or ham) gives the search no gradient,
and a **query limit** per sender caps the search. We repeat the greedy attack with (a) the score, (b) only the label, (c) a budget of queries.

In [ ]:
def label_only(victim):
    return al.Victim(
        victim.name + " (label only)", lambda docs: np.where(victim.score(docs) >= 0, 1.0, -1.0), encoder=victim.encoder
    )


def asr_with_budget(v, budget):
    wins = 0
    caught = 0
    for m in sub:
        if v.score([m])[0] < 0:
            continue
        caught += 1
        q0 = v.queries
        cur = m
        for _ in range(12):
            cands = al.op_append(CANDIDATES)(cur)
            room = budget - (v.queries - q0)
            if room <= 0:
                break
            cands = cands[:room]
            sc = v.score(cands)
            cur = cands[int(np.argmin(sc))]
            if sc.min() < 0:
                wins += 1
                break
    return wins / caught


print(f"{'API':34}" + "".join(f"{b:>8}" for b in (50, 150, 500)))
for k in ("nb_counts", "tfidf_lr"):
    for label, v in ((V[k].name + ", score", V[k]), (V[k].name + ", label only", label_only(V[k]))):
        print(f"{label:34}" + "".join(f"{asr_with_budget(v, b):8.2f}" for b in (50, 150, 500)))

With the label only, the greedy search has nothing to climb: it needs the same luck as random search. A query budget caps the score-based attack
too. Both raise the attacker's price; neither removes the weakness (a **surrogate** trained on the labels of many queries brings the score back: Tramèr et al., 2016).

## The arms race in one table

| Defence | Stops | Attacker's answer | Cost to the defender |
|:--|:--|:--|:--|
| label filter (A) | most poisoned labels | slower, subtler poisoning | a model per retraining |
| normalisation (B) | invisible characters, dots, spaces | attacks that add words (level 2) | almost none |
| adversarial training (C) | the attacks it has seen | adaptive search, new attacks | data, retraining, small F1 cost |
| ensemble (D) | one-model attacks | attack the ensemble | several models, more false positives |
| label-only API, query limit (E) | black-box search | surrogate model | a less informative API |

## Exercises

1. In A, repeat the trigger **three times** in each poisoned message. What happens to Naive Bayes, and why?
2. Add a fourth attack (synonyms with pre-trained Word2Vec, notebook 09) to the suite of C and re-run. Does adversarial training against the
   *other* three help against it?
3. Replace the OR rule of D by the *mean* of the standardised scores. Which is more robust, which produces fewer false positives?
4. Design an attack that beats normalisation B (hint: which characters does `normalise` not map?) and measure it.